In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/CAISI_phase2/SaLoRA-finetuning/SaLoRA-Repo

/content/drive/MyDrive/CAISI_phase2/SaLoRA-finetuning/SaLoRA-Repo


In [ ]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 25.9 M

In [ ]:
import numpy as np
import random
import torch
from datasets import load_dataset

In [ ]:
def get_alpaca(nsamples, seed, seqlen, tokenizer, disentangle=False, dataset="alpaca_cleaned_no_safety_train", modelname = ''):
    data_files = {"train": "lowrank_prune/data/alpaca_5k_train.csv"}
    traindata = load_dataset("csv", data_files=data_files, split="train")
    random.seed(seed)
    # Encode datasets
    trainloader = []
    if disentangle:
        traindata_sampled = traindata.shuffle(seed=seed).select(range(nsamples))
        count=0
        for i in range(nsamples):
            count+=1
            user_prompt = traindata_sampled["prompt"][i]
            assistant_response = traindata_sampled["response"][i]
            formatted_messages = [
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_response},
            ]

            encoded = tokenizer.apply_chat_template(
                formatted_messages,
                tokenize=True,
                return_tensors="pt",
                add_generation_prompt=False
            )
            inp = encoded.clone()
            text_without_response = tokenizer.apply_chat_template(
                formatted_messages[:-1],
                tokenize=True,
                return_tensors="pt",
                add_generation_prompt=True
            )
            prompt_len = text_without_response.shape[1]

            tar = inp.clone()
            tar[:, :prompt_len] = -100
            trainloader.append((inp, tar))
            if count<3:
              print("\n--- Example", i, "---")
              print("Input IDs shape:", inp.shape)
              print("Target IDs shape:", tar.shape)
              print("Prompt length:", prompt_len)
              print("Masked target positions:", (tar[0] == -100).nonzero(as_tuple=True)[0])
              print("Decoded input:", tokenizer.decode(inp[0], skip_special_tokens=False))

    else:
        trainenc = tokenizer(" ".join(traindata["text"]), return_tensors="pt")
        # Generate samples from training set
        for _ in range(nsamples):
            i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
            j = i + seqlen
            inp = trainenc.input_ids[:, i:j]
            tar = inp.clone()
            tar[:, :-1] = -100
            trainloader.append((inp, tar))
    return None, None

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")


In [ ]:
get_alpaca(128, 0, 2048, tokenizer, True, modelname='llama2')


--- Example 0 ---
Input IDs shape: torch.Size([1, 316])
Target IDs shape: torch.Size([1, 316])
Prompt length: 46
Masked target positions: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45])
Decoded input: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Create a script for a radio advertisement for a restaurant.:<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Background music starts playing**

Narrator: Are you tired of the same old meals at home, day in and day out? Do you crave a night out, surrounded by the warmth of good food and great company? Look no further!

**Sound of a sizzling pan**

Narrator: Introducing [Name of Restaurant], the perfect place to satisfy your hunger. Our menu feature

(None, None)